In [ ]:
import requests
from dotenv import load_dotenv
import boto3
from pathlib import Path
import pandas as pd
import json

load_dotenv()

s3 = boto3.client("s3")
bucket_name = "weather-data-eng"

# --- Load all three gold tables ---
attractions_df = pd.read_csv(s3.get_object(
    Bucket=bucket_name, Key="gold/fact_city_attractions.csv")["Body"])

by_category_df = pd.read_csv(s3.get_object(
    Bucket=bucket_name, Key="gold/fact_city_activities_by_category.csv")["Body"])

counts_df = pd.read_csv(s3.get_object(
    Bucket=bucket_name, Key="gold/fact_activity_counts.csv")["Body"])

tables = {
    "fact_city_attractions": attractions_df,
    "fact_city_activities_by_category": by_category_df,
    "fact_activity_counts": counts_df,
}

# --- Basic shape/structure check ---
for name, df in tables.items():
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("Nulls:\n", df.isnull().sum())
    print("------\n")

=== fact_city_attractions ===
Shape: (199, 9)
Columns: ['id', 'city', 'name', 'feature', 'dist', 'rate', 'kinds', 'lat', 'lon']
Nulls:
 id         0
city       0
name       0
feature    0
dist       0
rate       0
kinds      0
lat        0
lon        0
dtype: int64
------

=== fact_city_activities_by_category ===
Shape: (218, 7)
Columns: ['city', 'name', 'kind', 'rate', 'dist', 'lat', 'lon']
Nulls:
 city    0
name    0
kind    0
rate    0
dist    0
lat     0
lon     0
dtype: int64
------

=== fact_activity_counts ===
Shape: (31, 3)
Columns: ['city', 'kind', 'attraction_count']
Nulls:
 city                0
kind                0
attraction_count    0
dtype: int64
------



In [2]:
# --- Check attractions table: dedup confirmation ---
print("Total rows:", len(attractions_df))
print("Distinct (city, name) pairs:", attractions_df.drop_duplicates(subset=["city", "name"]).shape[0])
# these two numbers should match — confirms dedup worked

print("\nCities covered:", attractions_df["city"].nunique())
print(attractions_df["city"].value_counts())

Total rows: 199
Distinct (city, name) pairs: 199

Cities covered: 5
city
jakarta      50
guangzhou    45
shanghai     41
tokyo        38
mumbai       25
Name: count, dtype: int64
